In [0]:
from pyspark.sql.functions import (
    col, trim, upper, lower, when, regexp_replace,
    to_date, to_timestamp, current_timestamp, lit,
    coalesce, isnan, isnull
)
from pyspark.sql.types import DecimalType, IntegerType, BooleanType


In [0]:
# ===========================================================================
# CÉLULA 1 - Configurações
# ===========================================================================
CATALOG       = "workspace"
SCHEMA_BRONZE = "bronze"
SCHEMA_SILVER = "silver"

In [0]:
# ===========================================================================
# CÉLULA 2 - Criar schema SILVER
# ===========================================================================
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_SILVER}")
print(f"Schema '{SCHEMA_SILVER}' pronto.")

In [0]:
# ===========================================================================
# CÉLULA 3 - Funções auxiliares de Data Quality
# ===========================================================================
def log_dq(tabela, total_antes, total_depois, descricao=""):
    removidos = total_antes - total_depois
    pct = round(removidos / total_antes * 100, 2) if total_antes > 0 else 0
    print(f"  DQ [{tabela}]: {total_antes} → {total_depois} "
          f"(removidos: {removidos} | {pct}%) {descricao}")


In [0]:
# ===========================================================================
# CÉLULA 4 - Processar CLIENTES
# ===========================================================================
print("\n>>> Data Quality: CLIENTES")
df = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.clientes")
total_antes = df.count()

df_silver = (
    df
    # Limpa espaços e padroniza
    .withColumn("nome",   trim(col("nome")))
    .withColumn("email",  lower(trim(col("email"))))
    .withColumn("cidade", trim(col("cidade")))
    .withColumn("estado", upper(trim(col("estado"))))
    # Remove nulos obrigatórios
    .filter(col("id_cliente").isNotNull())
    .filter(col("nome").isNotNull() & (col("nome") != ""))
    .filter(col("email").isNotNull() & (col("email") != ""))
    # Valida formato de email (básico)
    .filter(col("email").contains("@"))
    # Normaliza booleano
    .withColumn("ativo", col("ativo").cast(BooleanType()))
    # Formata data
    .withColumn("data_cadastro", to_date(col("data_cadastro")))
    # Audit columns
    .withColumn("_silver_ts", current_timestamp())
    # Remove colunas de controle do Bronze
    .drop("_bronze_ingestion_ts", "_source_table", "_source_format")
)

log_dq("clientes", total_antes, df_silver.count(), "validação email + nulos")
df_silver.write.format("delta").mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA_SILVER}.clientes")

In [0]:
# ===========================================================================
# CÉLULA 5 - Processar CATEGORIAS
# ===========================================================================
print("\n>>> Data Quality: CATEGORIAS")
df = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.categorias")
total_antes = df.count()

df_silver = (
    df
    .withColumn("nome",      trim(col("nome")))
    .withColumn("descricao", trim(coalesce(col("descricao"), lit(""))))
    .filter(col("id_categoria").isNotNull())
    .filter(col("nome").isNotNull() & (col("nome") != ""))
    .withColumn("_silver_ts", current_timestamp())
    .drop("_bronze_ingestion_ts", "_source_table", "_source_format")
)

log_dq("categorias", total_antes, df_silver.count())
df_silver.write.format("delta").mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA_SILVER}.categorias")


In [0]:
# ===========================================================================
# CÉLULA 6 - Processar PRODUTOS
# ===========================================================================
print("\n>>> Data Quality: PRODUTOS")
df = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.produtos")
total_antes = df.count()

df_silver = (
    df
    .withColumn("nome",    trim(col("nome")))
    .withColumn("preco",   col("preco").cast(DecimalType(10, 2)))
    .withColumn("estoque", col("estoque").cast(IntegerType()))
    .withColumn("ativo",   col("ativo").cast(BooleanType()))
    # Regras de negócio: preço e estoque não podem ser negativos
    .filter(col("id_produto").isNotNull())
    .filter(col("preco").isNotNull() & (col("preco") >= 0))
    .filter(col("estoque").isNotNull() & (col("estoque") >= 0))
    .filter(col("id_categoria").isNotNull())
    .withColumn("_silver_ts", current_timestamp())
    .drop("_bronze_ingestion_ts", "_source_table", "_source_format")
)

log_dq("produtos", total_antes, df_silver.count(), "preço/estoque >= 0")
df_silver.write.format("delta").mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA_SILVER}.produtos")


In [0]:
# ===========================================================================
# CÉLULA 7 - Processar PEDIDOS
# ===========================================================================
print("\n>>> Data Quality: PEDIDOS")
df = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.pedidos")
total_antes = df.count()

STATUS_VALIDOS = ["pendente", "processando", "em_transito", "entregue", "cancelado"]

df_silver = (
    df
    .withColumn("status",      lower(trim(col("status"))))
    .withColumn("forma_pagto", lower(trim(col("forma_pagto"))))
    .withColumn("valor_total", col("valor_total").cast(DecimalType(12, 2)))
    .withColumn("data_pedido", to_timestamp(col("data_pedido")))
    # Valida status permitido
    .filter(col("status").isin(STATUS_VALIDOS))
    # Valor total deve ser positivo (exceto cancelados → podem ser 0)
    .filter(
        (col("status") == "cancelado") |
        (col("valor_total").isNotNull() & (col("valor_total") > 0))
    )
    .filter(col("id_cliente").isNotNull())
    .withColumn("_silver_ts", current_timestamp())
    .drop("_bronze_ingestion_ts", "_source_table", "_source_format")
)

log_dq("pedidos", total_antes, df_silver.count(), "status válidos + valor > 0")
df_silver.write.format("delta").mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA_SILVER}.pedidos")


In [0]:
# ===========================================================================
# CÉLULA 8 - Processar ITENS_PEDIDO
# ===========================================================================
print("\n>>> Data Quality: ITENS_PEDIDO")
df = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.itens_pedido")
total_antes = df.count()

df_silver = (
    df
    .withColumn("quantidade", col("quantidade").cast(IntegerType()))
    .withColumn("preco_unit", col("preco_unit").cast(DecimalType(10, 2)))
    # Adiciona coluna calculada
    .withColumn("valor_item",
        (col("quantidade") * col("preco_unit")).cast(DecimalType(12, 2)))
    # Regras: quantidade e preço devem ser > 0
    .filter(col("id_item").isNotNull())
    .filter(col("quantidade") > 0)
    .filter(col("preco_unit") > 0)
    .withColumn("_silver_ts", current_timestamp())
    .drop("_bronze_ingestion_ts", "_source_table", "_source_format")
)

log_dq("itens_pedido", total_antes, df_silver.count(), "qtd/preço > 0 + valor_item calculado")
df_silver.write.format("delta").mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA_SILVER}.itens_pedido")


In [0]:
# ===========================================================================
# CÉLULA 9 - Relatório final
# ===========================================================================
print("\n" + "="*50)
print("SILVER - Tabelas criadas:")
display(spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA_SILVER}"))

print("\nAmostra Silver - pedidos:")
display(spark.sql(f"""
    SELECT id_pedido, id_cliente, data_pedido, status, valor_total, forma_pagto
    FROM {CATALOG}.{SCHEMA_SILVER}.pedidos LIMIT 5
"""))